In [1]:
import requests
import pandas as pd
import numpy as np
import time
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
missing_values = ["NaN "]
df = pd.read_csv("../dataset/foodDeli/train.csv", na_values=missing_values)
test_df = pd.read_csv("../dataset/foodDeli/test.csv", na_values=missing_values)
submission_df = pd.read_csv("../dataset/foodDeli/Sample_Submission.csv")

In [3]:
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
df.columns = df.columns.str.strip()
df.rename(columns={'Time_taken(min)': 'Time_taken'}, inplace=True)

# Test & Submission
test_df['ID'] = test_df['ID'].astype(str).str.strip()
submission_df['ID'] = submission_df['ID'].astype(str).str.strip()
test_df = pd.merge(test_df, submission_df, on='ID', how='left')


df['Weather_conditions'] = df['Weather_conditions'].str.replace('conditions ', '', regex=True)
test_df['Weather_conditions'] = test_df['Weather_conditions'].str.replace('conditions ', '', regex=True)

# Ép Time_taken về dạng số thực (Train)
df['Time_taken'] = df['Time_taken'].astype(str).str.replace('(min)', '', regex=False).str.strip()
df['Time_taken'] = pd.to_numeric(df['Time_taken'])


coord_cols = [
    'Restaurant_latitude', 'Restaurant_longitude', 
    'Delivery_location_latitude', 'Delivery_location_longitude'
]
for col in coord_cols:
    df[col] = df[col].abs()
    test_df[col] = test_df[col].abs()


df.dropna(inplace=True)
test_df.dropna(inplace=True)


null_island_idx = df[
    (df['Restaurant_latitude'] < 1) | 
    (df['Restaurant_longitude'] < 1) |
    (df['Delivery_location_latitude'] < 1) |
    (df['Delivery_location_longitude'] < 1)].index

df.drop(index=null_island_idx, inplace=True)

print(f"Kích thước tập Train hiện tại: {df.shape}")
print(f"Kích thước tập Test hiện tại : {test_df.shape}")

Kích thước tập Train hiện tại: (38064, 20)
Kích thước tập Test hiện tại : (10291, 20)


In [4]:
tqdm.pandas(desc="Đang gọi API OSRM")

# Hàm tính khoảng cách đường bộ bằng OSRM
def get_osrm_distance(lat1, lon1, lat2, lon2):
    url = f"http://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}?overview=false"
    try:
        time.sleep(0.05) # Ngủ 0.05s để không bị block IP
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            data = response.json()
            return data['routes'][0]['distance'] / 1000.0 # Trả về km
        return np.nan
    except:
        return np.nan

print("tính khoảng cách đường bộ cho tập TRAIN...")
df['Distance_km'] = df.progress_apply(
    lambda row: get_osrm_distance(
        row['Restaurant_latitude'], row['Restaurant_longitude'],
        row['Delivery_location_latitude'], row['Delivery_location_longitude']
    ), axis=1
)

print("tính khoảng cách đường bộ cho tập TEST...")
test_df['Distance_km'] = test_df.progress_apply(
    lambda row: get_osrm_distance(
        row['Restaurant_latitude'], row['Restaurant_longitude'],
        row['Delivery_location_latitude'], row['Delivery_location_longitude']), axis=1)

tính khoảng cách đường bộ cho tập TRAIN...


Đang gọi API OSRM: 100%|██████████| 38064/38064 [4:35:22<00:00,  2.30it/s]  


tính khoảng cách đường bộ cho tập TEST...


Đang gọi API OSRM: 100%|██████████| 10291/10291 [1:14:04<00:00,  2.32it/s]


In [5]:
df.to_csv("../dataset/foodDeli_processed/train_temp_osrm.csv", index=False)
test_df.to_csv("../dataset/foodDeli_processed/test_temp_osrm.csv", index=False)
df = pd.read_csv("../dataset/foodDeli_processed/train_temp_osrm.csv")
test_df = pd.read_csv("../dataset/foodDeli_processed/test_temp_osrm.csv")

In [6]:
# Tính cho tập Train (df)
df['Order_Time_Full'] = pd.to_datetime(df['Order_Date'] + ' ' + df['Time_Orderd'])
df['Picked_Time_Full'] = pd.to_datetime(df['Order_Date'] + ' ' + df['Time_Order_picked'])
df['Preparation_Time'] = (df['Picked_Time_Full'] - df['Order_Time_Full']).dt.total_seconds() / 60.0
df['Preparation_Time'] = np.where(df['Preparation_Time'] < 0, df['Preparation_Time'] + 1440, df['Preparation_Time'])

test_df['Order_Time_Full'] = pd.to_datetime(test_df['Order_Date'] + ' ' + test_df['Time_Orderd'])
test_df['Picked_Time_Full'] = pd.to_datetime(test_df['Order_Date'] + ' ' + test_df['Time_Order_picked'])
test_df['Preparation_Time'] = (test_df['Picked_Time_Full'] - test_df['Order_Time_Full']).dt.total_seconds() / 60.0
test_df['Preparation_Time'] = np.where(test_df['Preparation_Time'] < 0, test_df['Preparation_Time'] + 1440, test_df['Preparation_Time'])

In [7]:
categorical_cols = ['Weather_conditions', 'Road_traffic_density', 'Type_of_order', 'Type_of_vehicle','Vehicle_condition', 'Festival', 'City']

# One-hot
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

# Chuyển boolean sang int
for col in df.columns:
    if df[col].dtype == 'bool':
        df[col] = df[col].astype(int)

for col in test_df.columns:
    if test_df[col].dtype == 'bool':
        test_df[col] = test_df[col].astype(int)

# CẤT GIỮ ID TẬP TEST TRƯỚC KHI XÓA CỘT THÔ
test_id = test_df['ID']

# Xóa các cột định danh và cột thô đã được trích xuất
# ĐÃ SỬA: Thêm 'ID' vào danh sách xóa
cols_to_drop = [
    'ID', 'Delivery_person_ID', 'multiple_deliveries', 
    'Restaurant_latitude', 'Restaurant_longitude',
    'Delivery_location_latitude', 'Delivery_location_longitude',
    'Order_Date', 'Time_Orderd', 'Time_Order_picked',
    'Order_Time_Full', 'Picked_Time_Full'
]

df.drop(columns=cols_to_drop, errors='ignore', inplace=True)
test_df.drop(columns=cols_to_drop, errors='ignore', inplace=True)

In [8]:
y_train = df['Time_taken']
X_train = df.drop(columns=['Time_taken'])

y_test = test_df['Time_taken']
X_test = test_df.drop(columns=['Time_taken'])

# Ép khuôn cột cho Test theo đúng thứ tự và số lượng của Train
X_train_cols = X_train.columns
X_test = X_test.reindex(columns=X_train_cols, fill_value=0)

In [9]:

Q1_dist = X_train['Distance_km'].quantile(0.25)
Q3_dist = X_train['Distance_km'].quantile(0.75)
lower_dist = Q1_dist - 1.5 * (Q3_dist - Q1_dist)
upper_dist = Q3_dist + 1.5 * (Q3_dist - Q1_dist)

# Tìm Index của ngoại lai và Drop khỏi X_train, y_train
outlier_idx = X_train[(X_train['Distance_km'] < lower_dist) | (X_train['Distance_km'] > upper_dist)].index

X_train = X_train.drop(outlier_idx)
y_train = y_train.drop(outlier_idx)

print(f"Kích thước TRAIN sau khi dọn ngoại lai: {X_train.shape} (Đã xóa {len(outlier_idx)} dòng)")

Kích thước TRAIN sau khi dọn ngoại lai: (37554, 22) (Đã xóa 510 dòng)


In [10]:
scaler = StandardScaler()
cols_to_scale = ['Delivery_person_Age', 'Delivery_person_Ratings', 'Distance_km', 'Preparation_Time']

# Fit trên Train, Transform trên Train và Test
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

In [11]:




train_data_final = pd.concat([X_train, y_train], axis=1)
test_data_final = pd.concat([test_id, X_test, y_test], axis=1)

train_file = "../dataset/foodDeli_processed/train_processed.csv"
test_file = "../dataset/foodDeli_processed/test_processed.csv"

train_data_final.to_csv(train_file, index=False, encoding='utf-8-sig')
test_data_final.to_csv(test_file, index=False, encoding='utf-8-sig')